In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO
import os

In [2]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
VALID_SEASONS = ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
BASE_URL = 'https://www.transfermarkt.de'


### Get all teams

In [ ]:
# Get all teams participating in DFB Pokal since 2014-15 season


for season in VALID_SEASONS:
    teams = []
    teams_links = []
    link = BASE_URL + f'dfb-pokal/startseite/pokalwettbewerb/DFB?saison_id={season}'
    response = requests.get(link, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')


    # Find all 'tr' tags with the class 'begegnungZeile'
    rows = soup.find_all('tr', class_='begegnungZeile')

    for row in rows:
        home_team = row.find('td', class_='verein-heim')
        away_team = row.find('td', class_='verein-gast')
        if home_team and away_team:
            home_team_title = home_team.find('a')['title']
            home_team_link = home_team.find('a')['href']
            away_team_title = away_team.find_all('a')[1]['title']
            away_team_link = away_team.find_all('a')[1]['href']
            teams_links.append(home_team_link[:-4])
            teams_links.append(away_team_link[:-4])
            teams.append(home_team_title)
            teams.append(away_team_title)

    if len(teams_links) == 0:
        print(f"No teams found for {season} season")
        continue
    else:
        print(f"Found {len(set(teams_links))} teams in {season} season")


    df = pd.DataFrame({'team': teams, 'link': teams_links, 'season': season})
    # Remove duplicates
    df = df.drop_duplicates(subset=['team', 'link'])
    # Reset index
    df = df.reset_index(drop=True)
    # Save to CSV
    df.to_csv(f'datasets/teams/{season}.csv', index=False)



Found 64 teams in 2014 season
Found 64 teams in 2015 season
Found 64 teams in 2016 season
Found 64 teams in 2017 season
Found 64 teams in 2018 season
Found 64 teams in 2019 season
Found 64 teams in 2020 season
Found 64 teams in 2021 season
Found 64 teams in 2022 season
Found 64 teams in 2023 season
Found 64 teams in 2024 season


In [40]:
# Path to the folder containing the CSV files
folder_path = 'datasets/teams/'

# List to store dataframes
dataframes = []

# Iterate through all files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        # Read the CSV file and append it to the list
        dataframes.append(pd.read_csv(file_path))

# Combine all dataframes into a single dataframe
combined_df = pd.concat(dataframes, ignore_index=False)

# remove duplicates
combined_df = combined_df.drop_duplicates(subset=['team'])
# Reset index
combined_df = combined_df.reset_index(drop=True)

# Display the combined dataframe
teams_df = combined_df.drop(columns=['season'])

teams_df.to_csv('datasets/teams/teams.csv', index=False)

In [5]:
# Get additional data for each team

teams_df = pd.read_csv('datasets/teams/teams.csv')
new_data = pd.DataFrame(columns=['link', 'stadium_name', 'total_capacity', 'lawn_heating', 'field_length', 'field_width'])

for index, row in teams_df.iterrows():
    print(f"{index}",end=' ')
    additional_data = {
        'link': '',
        'stadium_name': '',
        'total_capacity': '',
        'lawn_heating': '',
        'field_length': '',
        'field_width': ''
    }

    team_link = row['link']
    final_link = BASE_URL + team_link.replace('spielplan','stadion') + '2014'
    print(final_link)
    response = requests.get(final_link, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find('table', class_='profilheader')

    # Extract the data from the table
    additional_data['link'] = team_link
    additional_data['stadium_name'] = table.find('th', string='Stadionname:').find_next('td').text.strip()
    try:
        additional_data['total_capacity'] = table.find('th', string='Gesamtkapazität:').find_next('td').text.strip()
    except AttributeError:
        additional_data['total_capacity'] = ''
    try:
        additional_data['lawn_heating'] = table.find('th', string='Rasenheizung:').find_next('td').text.strip()
    except AttributeError:
        additional_data['lawn_heating'] = ''
    
    try:
        playing_surface = table.find('th', string='Spielfeldgröße:').find_next('td').text.strip()
        additional_data['field_length'] = playing_surface.split('x')[0].strip().split('m')[0]
        additional_data['field_width'] = playing_surface.split('x')[1].strip().split('m')[0]
    except AttributeError:
        additional_data['field_length'] = ''
        additional_data['field_width'] = ''

    # Append the data to the new dataframe
    new_data = pd.concat([new_data, pd.DataFrame([additional_data])], ignore_index=True)



0 https://www.transfermarkt.de/borussia-dortmund/stadion/verein/16/saison_id/2014
1 https://www.transfermarkt.de/vfl-wolfsburg/stadion/verein/82/saison_id/2014
2 https://www.transfermarkt.de/fc-bayern-munchen/stadion/verein/27/saison_id/2014
3 https://www.transfermarkt.de/arminia-bielefeld/stadion/verein/10/saison_id/2014
4 https://www.transfermarkt.de/sc-freiburg/stadion/verein/60/saison_id/2014
5 https://www.transfermarkt.de/tsg-1899-hoffenheim/stadion/verein/533/saison_id/2014
6 https://www.transfermarkt.de/borussia-monchengladbach/stadion/verein/18/saison_id/2014
7 https://www.transfermarkt.de/bayer-04-leverkusen/stadion/verein/15/saison_id/2014
8 https://www.transfermarkt.de/vfr-aalen/stadion/verein/83/saison_id/2014
9 https://www.transfermarkt.de/1-fc-kaiserslautern/stadion/verein/2/saison_id/2014
10 https://www.transfermarkt.de/sg-dynamo-dresden/stadion/verein/129/saison_id/2014
11 https://www.transfermarkt.de/1-fc-koln/stadion/verein/3/saison_id/2014
12 https://www.transfermark

In [7]:
# Merge the new data with teams_df on the 'link' column
updated_teams_df = teams_df.merge(new_data, on='link', how='left')

# Display the updated dataframe
updated_teams_df

# Save the updated dataframe to a CSV file
updated_teams_df.to_csv('datasets/teams/teams.csv', index=False)

### Team Schedules: TODO

In [ ]:
# team schedule
link = f'https://www.transfermarkt.de/borussia-dortmund/spielplan/verein/16/saison_id/{VALID_SEASONS[0]}'

# Fetch the webpage using requests
response = requests.get(link, headers=HEADERS)

# Parse the page source with BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# Find all divs with class 'box'
box_divs = soup.find_all('div', class_='box')

# Iterate through the divs to find the one with the specific h2 tag
for div in box_divs:
    h2_tag = div.find('h2', class_='content-box-headline content-box-headline--inverted content-box-headline--logo content-box-headline--bottom-bordered content-box-headline--extra-space')
    if h2_tag:
        box_div = div
        break

# Find the table within the div
t_section = box_div.find('div', class_='responsive-table')
table = t_section.find('table')

# Convert the HTML table into a pandas DataFrame
df = pd.read_html(StringIO(str(table)))[0]

df = df.drop(columns=['Gegner'])


df.set_index('Spieltag', inplace=True)

# Rename the columns
df.columns = ['Date', 'Time', 'H/A', 'Ranking', 'Opponent', 'Formation', 'Spectators', 'Result']

# Display the DataFrame
df


,Date,Time,H/A,Ranking,Opponent,Formation,Spectators,Result
Spieltag,,,,,,,,
1,Sa. 15.08.2015,18:30,H,(2.),Bor. M'gladbach (4.),4-1-4-1,81.359,4:0
2,So. 23.08.2015,15:30,A,(2.),FC Ingolstadt (8.),4-1-4-1,15.000,0:4
3,So. 30.08.2015,15:30,H,(1.),Hertha BSC (7.),4-1-4-1,80.500,3:1
4,Sa. 12.09.2015,15:30,A,(1.),Hannover 96 (16.),4-1-4-1,49.000,2:4
5,So. 20.09.2015,17:30,H,(1.),B. Leverkusen (13.),4-1-4-1,81.359,3:0
6,Mi. 23.09.2015,20:00,A,(1.),TSG Hoffenheim (15.),4-1-4-1,29.000,1:1
7,So. 27.09.2015,17:30,H,(2.),Darmstadt 98 (10.),4-1-4-1,81.359,2:2
8,So. 04.10.2015,17:30,A,(2.),Bayern München (1.),4-3-1-2,75.000,5:1
9,Fr. 16.10.2015,20:30,A,(2.),1.FSV Mainz 05 (8.),4-1-4-1,34.000,0:2
